# LLP3 · PHASE 2 · DEFINE
## Problem Scoping & Persistence Baseline

Phase 2 turns your Phase 1 domain into a precise forecasting problem and establishes the reference model that later work must improve upon.

This notebook is a **starter scaffold, not a worked solution**.

It provides:
- the fixed domain constraints;
- safe dataset loading;
- methodological guardrails;
- guiding questions;
- checkpoints for your own implementation.

Your team is responsible for constructing the future target, designing the chronological evaluation split, implementing the persistence baseline, calculating the metrics, and interpreting the results.


Prediction of CO2 concentration in indoor environment so that the ventitalltion system can regulate the air quality automatically. 

## 1. Continue With Your Selected Domain

Use the same project domain selected in Phase 1.

| Domain | Full Project Dataset | Forecasting Variable | CSV / Code Field | Horizon |
|---|---|---|---|---|
| Environmental | `environmental_project.csv` | CO₂ concentration | `co2_ppm` | 30 minutes = 6 intervals |
| Weather | `weather_project.csv` | Solar radiation | `solar_radiation_w_m2` | 60 minutes = 12 intervals |

The supplied datasets use a 5-minute cadence.

**Environmental rule:** temporal operations must respect `room_id`. A future target must never shift values across room boundaries.

**Phase 2 scope:** establish the forecasting problem and the persistence benchmark. Do not begin the Phase 3 model tournament here.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DOMAIN = "environmental"   # change to "weather" if required

FILES = {
    "environmental": "ines_1week.csv",
    "weather": "weather_project.csv",
}

if DOMAIN not in FILES:
    raise ValueError("DOMAIN must be 'environmental' or 'weather'.")

target_file = FILES[DOMAIN]

candidates = [
    Path(target_file),
    Path("../dataset") / target_file,
    Path("dataset") / target_file,
]

data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(f"Could not find {target_file}")

df = pd.read_csv(data_path)
df["time"] = pd.to_datetime(df["time"], utc=True)

print("Domain:", DOMAIN)
print("Loaded:", data_path)
print("Shape:", df.shape)

df.head()


Domain: environmental
Loaded: ines_1week.csv
Shape: (40249, 9)


,time,temperature,humidity,co2,pm25,pm10,tvoc,sound,pressure
0,2026-10-21 13:22:52.482036+00:00,22.366,51.098,492.5,2.18,2.51,130.4,72.93,992.83
1,2026-10-21 13:23:07.447125+00:00,22.367,51.058,495.9,2.19,2.47,129.7,73.51,992.85
2,2026-10-21 13:23:22.460616+00:00,22.396,51.130,497.2,2.21,2.49,130.8,73.92,992.83
3,2026-10-21 13:23:37.474049+00:00,22.415,51.200,494.1,2.25,2.41,131.6,73.16,992.84
4,2026-10-21 13:23:52.530097+00:00,22.455,51.242,493.1,2.21,2.40,131.2,72.96,992.85


## 2. Confirm the Forecasting Context

Before constructing a target, reconnect the Phase 2 dataset to what you learned in Phase 1.

Check:

- the available columns and their data types;
- the timestamp range;
- missing values relevant to forecasting;
- the target variable;
- grouping structure such as `room_id`;
- any Phase 1 findings that could affect forecasting.

### Guiding questions

1. Which variables would actually be available at prediction time?

Past values of all environmental features recorded up to timestamp 't' (temperature, humidity, co2, pm25, pm10, tvoc, sound, and pressure).

2. Which fields are identifiers rather than predictors?

time: Functions strictly as the chronological timestamp index rather than a direct numerical predictor for modeling.

3. Are there missing measurements that affect target construction or evaluation?

An inspection of the dataset reveals 0 missing values across all 40,249 rows and 9 columns, meaning data completeness is ideal for constructing the future target and evaluating baselines.

4. Does the Environmental dataset need room-specific handling?

No room-specific cross-boundary handling is required because this file represents a single continuous monitoring session for room INES.

5. Which Phase 1 observations should influence your Phase 2 decisions?

The dataset spans 7 days (from October 21 to October 28, 2026) sampled at a dense 15-second cadence. Because short-term fluctuations are prominent at this frequency, Phase 2 and subsequent modeling must account for temporal autocorrelation, rolling historical lags, or smoothing when forecasting CO2 concentration 30 minutes ahead. 

In [3]:
# TODO: Inspect the Phase 2 dataset sufficiently to justify your forecasting setup.
#
# Your evidence should support the decisions you make later in this notebook.
#
# Your code here:

import pandas as pd

# 1. Load the dataset
df = pd.read_csv('ines_1week.csv')

# 2. Inspect data types and missing values
print("--- Dataset Info ---")
print(df.info())
print("\n--- Missing Values ---")
print(df.isnull().sum())

# 3. Inspect timestamp range and frequency
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time')
print("\n--- Timestamp Range ---")
print(f"Start: {df['time'].min()} | End: {df['time'].max()}")

print("\n--- Time Interval Statistics ---")
print(df['time'].diff().describe())


--- Dataset Info ---
<class 'pandas.DataFrame'>
RangeIndex: 40249 entries, 0 to 40248
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   time         40249 non-null  str    
 1   temperature  40249 non-null  float64
 2   humidity     40249 non-null  float64
 3   co2          40249 non-null  float64
 4   pm25         40249 non-null  float64
 5   pm10         40249 non-null  float64
 6   tvoc         40249 non-null  float64
 7   sound        40249 non-null  float64
 8   pressure     40249 non-null  float64
dtypes: float64(8), str(1)
memory usage: 2.8 MB
None

--- Missing Values ---
time           0
temperature    0
humidity       0
co2            0
pm25           0
pm10           0
tvoc           0
sound          0
pressure       0
dtype: int64

--- Timestamp Range ---
Start: 2026-10-21 13:22:52.482036+00:00 | End: 2026-10-28 13:22:25.292634+00:00

--- Time Interval Statistics ---
count                     40248
mean  

## 3. Define the Forecasting Problem

State the problem **before** implementing it.

Your problem statement should identify:

- selected domain;
- forecasting variable;
- CSV / code target field;
- physical unit;
- forecast horizon;
- number of 5-minute intervals ahead;
- operational relevance;
- what information is available at prediction time;
- the main leakage risk you must avoid.

### Team Problem Statement

- Selected Domain: Indoor Air Quality & Environmental Monitoring
- Forecasting Variable: Carbon Dioxide (CO2) concentration
- CSV / Code Target Field: CO2
- Physical Unit: ppm (parts per million)
- Forecast Horizon: 30 minutes ahead
- Number of 5-Minute Intervals Ahead: 6 intervals (since data is recorded at a 15-second cadence, 30 minutes equals 120 individual 15-second steps or 6 standard 5-minute blocks)
- Operational Relevance: Proactive ventilation management to automatically regulate air quality, maintain occupant comfort, and prevent unhealthy CO2 buildup in indoor spaces.
- Available Information at Prediction Time: Historical sensor measurements (temperature, humidity, CO2, PM{2.5}, PM{10}, TVOC, sound, and pressure) and derived calendar features up to current timestamp 't'.
- Main Leakage Risk to Avoid: Using future target values or improperly aligned lag features that incorporate observations beyond the prediction timestamp 't'.



## 4. Construct the Future Target

Create a future label representing:

\[
y(t+h)
\]

Fixed course constraints:

- Environmental: CO₂ concentration 30 minutes ahead = **6 intervals**.
- Weather: solar radiation 60 minutes ahead = **12 intervals**.
- Environmental shifts must remain within each `room_id`.
- Future information must not enter the predictor set.

### Guiding questions

1. What must be sorted before shifting?

The dataframe must be sorted chronologically by the timestamp column (time) to ensure that shifting pulls actual future observations rather than arbitrary row orders.

2. Which column should be shifted?

The target variable column, which is co2 (resulting in the output column named target_future).

3. In which direction should the shift operate?

It should shift upward using a negative offset (e.g., shift(-120) for 30 minutes ahead at a 15-second cadence), so that the row at time 't' receives the CO2 value from 30 minutes into the future (t + h).

4. How will you prevent Environmental rows from crossing `room_id` boundaries?

Since this dataset represents a single continuous monitoring session for room INES without multiple rooms or a room_id column, standard chronological shifting works directly on the sorted dataframe. If multi-room data is used elsewhere, shifting must be grouped by room_id using groupby.

5. Which rows become unusable after target construction, and why?

The last 'h' rows (the final 120 rows) become unusable because there are no future observations available beyond the end of the dataset to look ahead into. These rows contain NaN for target_future and must be dropped before model training.

In [6]:
# TODO: Construct the future target.
#
# Suggested output column name:
# target_future
#
# Your code here:

import pandas as pd

# 1. Load and sort the dataset chronologically
df = pd.read_csv('ines_1week.csv')
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

# 2. Construct the future target for 30 minutes ahead at 15-second intervals
# 30 minutes * (60 seconds / 15 seconds) = 120 intervals
h = 120  
df['target_future'] = df['co2'].shift(-h)

# 3. Drop rows with missing target values at the end of the series
df_model = df.dropna(subset=['target_future']).copy()

print(f"Original shape: {df.shape} | Cleaned shape for modeling: {df_model.shape}")


Original shape: (5755, 10) | Cleaned shape for modeling: (5635, 10)


## 5. Design a Chronological Train / Validation / Test Split

Time-series forecasting must preserve temporal order.

Create chronological periods for:

- training;
- validation;
- test.

Do **not** randomly shuffle the time series.

### Your split should satisfy

- training observations occur before validation observations;
- validation observations occur before test observations;
- preprocessing or model-development choices are not learned from the test period;
- Environmental handling remains consistent across `room_id` groups.

### Student task

Choose and justify the split boundaries. There is no single required percentage split, but your design must be defensible and reproducible.


To maintain strict temporal integrity without data leakage, we divide the cleaned modeling dataset chronologically into standard proportions (70% training, 15% validation, and 15% test):
Training Set (70%): Oldest chronological window, used for model training and pattern learning. 
Validation Set (15%): Intermediate window, used for hyperparameter tuning, model selection, and development-time evaluation. 
Test Set (15%): Most recent chronological window, held out entirely to report final unbiased model generalization performance.   

In [4]:

import pandas as pd

# 1. Load and sort the dataset chronologically
df = pd.read_csv('ines_1week.csv')
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

# 2. Construct future target (30 minutes ahead = 120 intervals for 15s data)
h = 120
df['target_future'] = df['co2'].shift(-h)
df_model = df.dropna(subset=['target_future']).copy()

# 3. Define chronological split ratios (70% train, 15% validation, 15% test)
n = len(df_model)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_model.iloc[:train_end].copy()
val_df = df_model.iloc[train_end:val_end].copy()
test_df = df_model.iloc[val_end:].copy()

# 4. Verify row counts and time ranges
print(f"Training Set:   {len(train_df)} rows | {train_df['time'].min()} to {train_df['time'].max()}")
print(f"Validation Set: {len(val_df)} rows | {val_df['time'].min()} to {val_df['time'].max()}")
print(f"Test Set:       {len(test_df)} rows | {test_df['time'].min()} to {test_df['time'].max()}")

Training Set:   28090 rows | 2026-10-21 13:22:52.482036+00:00 to 2026-10-26 10:37:46.770262+00:00
Validation Set: 6019 rows | 2026-10-26 10:38:01.787398+00:00 to 2026-10-27 11:46:09.710323+00:00
Test Set:       6020 rows | 2026-10-27 11:46:24.724920+00:00 to 2026-10-28 12:52:23.855493+00:00


## 6. Implement the Persistence Baseline

Persistence assumes that the future target will remain equal to its current observed value:

\[
\hat{y}(t+h) = y(t)
\]

This is the reference model for the project.

Later machine-learning models are only useful if they provide meaningful improvement over this baseline.

### Guiding questions


1. Which current measurement corresponds to the future target?

The current CO2 concentration measurement at timestamp 't'

2. Does persistence require model fitting?

No, the persistence model is a naive reference baseline that requires zero training, parameter estimation, or optimization.

3. Which rows can be evaluated validly?

All rows within the validation (val_df) and test (test_df) sets that have a corresponding non-null target_future (i.e., excluding rows at the very end of the dataset where future values cannot be looked up).

4. Why might persistence be strong for slowly changing physical systems?

In indoor environments where CO2 changes gradually due to physical inertia over short intervals, the current value is already very close to the value shortly ahead, making it a robust and difficult baseline to beat.

5. Under what physical conditions might it fail?

It fails during abrupt environmental transitions or sudden events, such as a sudden influx of people entering a room (causing rapid CO2 spikes) or sudden ventilation/HVAC activations.

In [7]:

from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# 1. Implement persistence predictions: y_hat(t+h) = y(t)
# In our dataframe, current co2 is y(t)
val_df['pred_persistence'] = val_df['co2']
test_df['pred_persistence'] = test_df['co2']

# 2. Evaluate performance on validation set
val_rmse = np.sqrt(mean_squared_error(val_df['target_future'], val_df['pred_persistence']))
val_mae = mean_absolute_error(val_df['target_future'], val_df['pred_persistence'])

print(f"Validation Persistence Baseline:")
print(f" * RMSE: {val_rmse:.4f} ppm")
print(f" * MAE:  {val_mae:.4f} ppm")

# 3. Evaluate performance on test set
test_rmse = np.sqrt(mean_squared_error(test_df['target_future'], test_df['pred_persistence']))
test_mae = mean_absolute_error(test_df['target_future'], test_df['pred_persistence'])

print(f"\nTest Persistence Baseline:")
print(f" * RMSE: {test_rmse:.4f} ppm")
print(f" * MAE:  {test_mae:.4f} ppm")


Validation Persistence Baseline:
 * RMSE: 23.5983 ppm
 * MAE:  19.3930 ppm

Test Persistence Baseline:
 * RMSE: 20.7250 ppm
 * MAE:  16.5634 ppm


## 7. Evaluate the Baseline

Calculate:

- MAE;
- RMSE;
- R².

Use the validation set for development-time understanding and report final baseline performance on the test set.

Only evaluate rows for which both the prediction and future target are valid.

Do not silently discard rows. Record how many rows are evaluated and how many are excluded.

### Student task

Implement the evaluation and make sure the metrics are interpreted in the physical unit of the forecasting variable.


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# TODO:
# - prepare y_true and persistence predictions;
# - handle invalid / unavailable rows explicitly;
# - calculate MAE, RMSE, and R²;
# - record evaluated and excluded row counts.
#
# Your code here:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# 1. Load and prepare dataset
df = pd.read_csv('ines_1week.csv')
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

# 2. Construct future target (h = 120 intervals for 30 min ahead at 15s frequency)
h = 120
total_initial_rows = len(df)
df['target_future'] = df['co2'].shift(-h)

# Track global excluded rows from target construction
df_model = df.dropna(subset=['target_future']).copy()
global_excluded_rows = total_initial_rows - len(df_model)

# 3. Chronological split (70% train, 15% val, 15% test)
n = len(df_model)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_model.iloc[:train_end].copy()
val_df = df_model.iloc[train_end:val_end].copy()
test_df = df_model.iloc[val_end:].copy()

# 4. Implement Persistence Predictions: y_hat(t+h) = y(t)
val_df['pred_persistence'] = val_df['co2']
test_df['pred_persistence'] = test_df['co2']

# 5. Evaluation function with explicit row tracking and physical unit interpretation
def evaluate_baseline_subset(subset_df, subset_name="Validation"):
    # Filter only valid rows where both prediction and target are present
    valid_subset = subset_df.dropna(subset=['target_future', 'pred_persistence'])
    
    n_evaluated = len(valid_subset)
    n_excluded = len(subset_df) - n_evaluated
    
    y_true = valid_subset['target_future']
    y_pred = valid_subset['pred_persistence']
    
    # Calculate metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"================ {subset_name} Set Evaluation ================")
    print(f" Evaluated Rows : {n_evaluated}")
    print(f" Excluded Rows  : {n_excluded}")
    print(f" MAE            : {mae:.4f} ppm")
    print(f" RMSE           : {rmse:.4f} ppm")
    print(f" R² Score       : {r2:.4f}\n")

# Run evaluation on validation and test sets
evaluate_baseline_subset(val_df, subset_name="Validation")
evaluate_baseline_subset(test_df, subset_name="Test")

print(f"Global Excluded Rows (due to target horizon at end of dataset): {global_excluded_rows}")


================ Validation Set Evaluation ================
 Evaluated Rows : 6019
 Excluded Rows  : 0
 MAE            : 19.3930 ppm
 RMSE           : 23.5983 ppm
 R² Score       : 0.7631

================ Test Set Evaluation ================
 Evaluated Rows : 6020
 Excluded Rows  : 0
 MAE            : 16.5634 ppm
 RMSE           : 20.7250 ppm
 R² Score       : 0.8980

Global Excluded Rows (due to target horizon at end of dataset): 120


## 8. Create the Required Phase 2 Result CSV

Create:

`<domain>_phase_2_baseline_results.csv`

where `<domain>` is exactly:

- `environmental`, or
- `weather`.

Your result table should contain enough information to identify:

- domain;
- target;
- forecast horizon;
- evaluation split;
- number of evaluated rows;
- MAE;
- RMSE;
- R².

The starter notebook intentionally does **not** build this table for you.


In [10]:
output_filename = f"{DOMAIN}_phase_2_baseline_results.csv"

# TODO:
# 1. Build the baseline-results DataFrame.
# 2. Verify that it contains the required evidence.
# 3. Export it using output_filename.
#
# Your code here:

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Define domain and output filename
DOMAIN = "environmental"
output_filename = f"{DOMAIN}_phase_2_baseline_results.csv"

# 2. Load and sort the INES dataset chronologically
df = pd.read_csv('ines_1week.csv')
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time').reset_index(drop=True)

# 3. Construct future target for 30 minutes ahead (15-second cadence -> 120 intervals)
h = 120  
df['target_future'] = df['co2'].shift(-h)
df_model = df.dropna(subset=['target_future']).copy()

# 4. Chronological Split (70% train, 15% validation, 15% test)
n = len(df_model)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df_model.iloc[:train_end].copy()
val_df = df_model.iloc[train_end:val_end].copy()
test_df = df_model.iloc[val_end:].copy()

# 5. Persistence Predictions: y_hat(t+h) = y(t)
val_df['pred_persistence'] = val_df['co2']
test_df['pred_persistence'] = test_df['co2']

# 6. Helper function to compile evaluation metrics
def compute_metrics_dict(subset_df, split_name):
    valid_subset = subset_df.dropna(subset=['target_future', 'pred_persistence'])
    y_true = valid_subset['target_future']
    y_pred = valid_subset['pred_persistence']
    
    return {
        'domain': DOMAIN,
        'target': 'co2',
        'forecast_horizon': '120 intervals (30 min)',
        'evaluation_split': split_name,
        'number_of_evaluated_rows': len(valid_subset),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred)
    }

# 7. Build the results DataFrame
results_df = pd.DataFrame([
    compute_metrics_dict(val_df, 'validation'),
    compute_metrics_dict(test_df, 'test')
])

print("=== INES Dataset Baseline Results Summary ===")
print(results_df.to_string(index=False))

# 8. Export to CSV
results_df.to_csv(output_filename, index=False)
print(f"\nSuccessfully saved INES results to {output_filename}")

=== INES Dataset Baseline Results Summary ===
       domain target       forecast_horizon evaluation_split  number_of_evaluated_rows       MAE      RMSE       R2
environmental    co2 120 intervals (30 min)       validation                      6019 19.392989 23.598290 0.763150
environmental    co2 120 intervals (30 min)             test                      6020 16.563372 20.724989 0.897979

Successfully saved INES results to environmental_phase_2_baseline_results.csv


## 9. Interpret the Persistence Baseline

Answer in your own words and use evidence from your results.

1. What physical assumption does persistence make?
2. Where does persistence work well?
3. Where does it fail?
4. How large is MAE in the physical unit of the target?
5. What does RMSE reveal about larger errors?
6. What does R² add to your interpretation?
7. Are there room-specific or time-specific patterns worth investigating later?
8. What should Phase 3 attempt to improve?
9. What must remain unchanged in Phase 3 so that comparisons stay fair?

### Team Interpretation

*Write your evidence-based interpretation here.*


1. What physical assumption does persistence make?
Persistence assumes that the indoor environment is stationary and exhibits strong short-term physical inertia—meaning the $\text{CO}_2$ concentration 30 minutes into the future will remain exactly equal to its current observed value: $\hat{y}(t+h) = y(t)$.
3. Where does persistence work well?
It performs well during steady-state conditions when room occupancy is stable and $\text{CO}_2$ levels change gradually over time, resulting in smaller short-term prediction errors.
4. Where does it fail?
It fails during rapid environmental transitions, such as sudden occupancy changes (e.g., people entering or leaving a room, causing sharp $\text{CO}_2$ generation spikes) or abrupt HVAC/ventilation system activations.
5. How large is MAE in the physical unit of the target?
On the validation set, the Mean Absolute Error (MAE) is $19.39\text{ ppm}$, and on the test set, it is $16.56\text{ ppm}$, representing the average magnitude of absolute prediction errors in parts per million.
6. What does RMSE reveal about larger errors?
The Root Mean Square Error is $23.60\text{ ppm}$ (validation) and $20.73\text{ ppm}$ (test). Because RMSE squares errors before averaging, its higher value compared to MAE indicates that the baseline occasionally incurs larger, penalized errors during sudden $\text{CO}_2$ swings.
7. What does $R^2$ add to your interpretation?
The validation $R^2$ is $0.7631$, and the test $R^2$ is $0.8980$. Unlike smaller datasets, this longer 7-day monitoring series exhibits strong diurnal patterns where the current $\text{CO}_2$ value strongly correlates with the future target, allowing persistence to explain a high percentage of variance. However, it still lacks the ability to anticipate trend reversals.
8. Are there room-specific or time-specific patterns worth investigating later?
Yes. Because $\text{CO}_2$ levels are strongly tied to human presence and diurnal activity cycles, incorporating time-of-day features, rolling historical lags, and temporal interaction effects will be critical in Phase 3.
9. What should Phase 3 attempt to improve?
Phase 3 should build advanced machine learning models (e.g., autoregressive features, rolling statistics, and gradient boosting or regression models) to capture non-linear trends and respond faster to rapid $\text{CO}_2$ fluctuations than a naive persistence baseline.
10. What must remain unchanged in Phase 3 so that comparisons stay fair?
The exact target construction ($h = 120$ intervals / 30-minute horizon), the chronological train/validation/test split boundaries, and the evaluation metrics (MAE, RMSE, $R^2$) must remain strictly identical to ensure a valid and fair comparison.

Is it a good model?

As a Baseline: Yes, it is a surprisingly strong and robust baseline. Because indoor environments change gradually under normal conditions, simply copying the current value ($\hat{y}(t+h) = y(t)$) is difficult to beat. As an Operational Solution: 
It has clear limitations. Because persistence is purely reactive, it cannot anticipate trend reversals, sudden influxes of people, or automated ventilation adjustments.

## 10. Completion Check

Before submission, confirm that your notebook contains:

- a precise forecasting problem statement;
- the correct target and forecast horizon;
- leakage-safe future-target construction;
- room-safe target construction for Environmental data;
- chronological train / validation / test periods;
- a persistence baseline;
- validation and test evidence where appropriate;
- MAE, RMSE, and R²;
- evaluated / excluded row counts;
- physical interpretation of the errors;
- `<domain>_phase_2_baseline_results.csv`.

The dedicated Phase 2 assignment page remains authoritative for submission requirements.

**Remember:** this notebook gives you the structure and methodological constraints. Your team must implement and defend the analysis.
